# Initial Device Performance: Physics of the J-V Curve

## 1. Abstract
This notebook focuses on predicting the initial Power Conversion Efficiency ($PCE$) of perovskite devices. We analyze how the machine learning model captures the complex interplay between materials, stack architecture, and interface quality by mapping experimental data to fundamental photovoltaic parameters.

## 2. Theoretical Framework: The Physics-to-ML Mapping

### 2.1. The Shockley-Queisser (SQ) Limit and Band Gap ($E_g$)
The maximum theoretical efficiency of a single-junction solar cell is governed by the band gap ($E_g$) through the Shockley-Queisser limit. The short-circuit current ($J_{sc}$) is bounded by the number of photons with energy $h
u > E_g$:
$$
J_{sc}^{max} = q \int_{0}^{\lambda_g} \Phi_{AM1.5G}(\lambda) d\lambda
$$
Where $\Phi_{AM1.5G}(\lambda)$ is the solar photon flux. Perovskites are tunable from $\approx 1.2$ to $2.3$ eV.

*   **Physics Role:** $E_g$ determines the trade-off between $V_{oc}$ (which increases with $E_g$) and $J_{sc}$ (which decreases with $E_g$).
*   **ML Connection:** $E_g$ serves as the **Primary Splitting Feature**. The CatBoost model learns a non-linear 'Bell Curve' for efficiency, effectively approximating the SQ manifold. It identifies that the highest efficiencies cluster in the $1.5\text{--}1.6\text{ eV}$ range for single junctions, matching physical theory without explicit SQ equations.

### 2.2. Fill Factor ($FF$) and Interface Quality
The fill factor is the ratio of maximum power to the product of $V_{oc}$ and $J_{sc}$:
$$
FF = \frac{P_{max}}{V_{oc} \cdot J_{sc}} = \frac{V_{mp} \cdot J_{mp}}{V_{oc} \cdot J_{sc}}
$$
It is sensitive to series resistance ($R_s$) and shunt resistance ($R_{sh}$), which are dictated by transport layer conductivity and film morphology.

*   **Physics Role:** ETL/HTL interfaces determine the 'Interface Recombination Velocity'. Misaligned bands create barriers that reduce $FF$ and $J_{sc}$.
*   **ML Connection:** This provides **Structural Context**. By combining stoichiometry (band positions) with transport layer sequences, the model learns to penalize combinations that likely lead to high $R_s$ or poor extraction efficiency, acting as a 'surrogate' for the drift-diffusion equations.

### 2.3. Active Area ($Area$) and Scaling Laws
The Efficiency ($\eta$) is defined as:
$$
\eta = \frac{V_{oc} \cdot J_{sc} \cdot FF}{P_{in}}
$$
Experimentally, $\eta$ decreases as $Area$ increases due to lateral resistive losses in the TCO and increased defect density in large films.

*   **Physics Role:** Scaling induces non-uniformity and resistive drops.
*   **ML Connection:** Area acts as a **Scaling Bias**. The model learns to 'discount' the predicted PCE for large-area records, capturing the historical performance gap between lab-scale 'hero cells' ($< 0.1\text{ cm}^2$) and mini-modules ($> 10\text{ cm}^2$).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.metrics import r2_score, mean_absolute_error

pio.templates.default = "plotly_white"

# Load data
df_s = pd.read_parquet('../data/enriched_solar_panels.parquet').dropna(subset=['jv_default_pce'])
df_p = pd.read_parquet('../data/enriched_perovskites.parquet')

# Merge to get material features
df = pd.merge(df_s, df_p, on='composition_long_form', suffixes=('', '_mat'))

# Load model
model = joblib.load('../ml_models/initial_pce.joblib')

# Fix categorical columns for CatBoost dynamically
cat_features = [model.feature_names_[i] for i in model.get_cat_feature_indices()]
for col in cat_features:
    if col in df.columns:
        df[col] = df[col].fillna("None").astype(str)

X = df[model.feature_names_]
df['predicted_pce'] = model.predict(X)
df['residual'] = df['jv_default_pce'] - df['predicted_pce']

print(f"Predictions complete for {len(df)} devices. R2: {r2_score(df['jv_default_pce'], df['predicted_pce']):.3f}")

## 3. Data Distribution: Transport Layer Coverage
We visualize the frequency of ETL sequences to understand the model's 'experience' with different electron-extracting materials. 

**Observation:** TiO2 and SnO2 dominate the ETL landscape. The model's predictions for novel ETLs like organic small molecules or binary oxides may have higher uncertainty due to fewer training points.

In [ ]:
fig = px.bar(df['etl_stack_sequence'].value_counts().head(15).reset_index(), x='etl_stack_sequence', y='count', 
             title="Frequency of Common ETL Stack Sequences", 
             labels={'etl_stack_sequence': 'ETL Sequence', 'count': 'Sample Count'})
fig.update_layout(title_x=0.5, xaxis_tickangle=-45)
fig.show()

## 4. Feature Importance: Identifying the Performance Drivers
The CatBoost model ranks features based on their contribution to the prediction error reduction. This ranking helps us identify which physical parameters—be it the band gap, the specific transport layer sequence, or the cell architecture—most significantly influence the Power Conversion Efficiency ($PCE$).

In [ ]:
# Extract feature importance from CatBoost
importances = model.get_feature_importance()
feature_names = model.feature_names_
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

fig = px.bar(
    importance_df.head(15), 
    x='Importance', 
    y='Feature', 
    orientation='h',
    title="Top 15 Feature Importances (CatBoost)",
    labels={'Importance': 'Relative Importance', 'Feature': 'Feature Name'}
)
fig.update_layout(yaxis={'categoryorder':'total ascending'}, title_x=0.5)
fig.show()

## 5. Model Performance: Parity Plot and Error Metrics
A Parity Plot compares the experimental Power Conversion Efficiency (PCE) against the values predicted by the CatBoost model. This visualization is critical for assessing the model's reliability across the entire performance range. We also calculate the Coefficient of Determination ($R^2$) and Mean Absolute Error (MAE) to provide a quantitative measure of accuracy.

In [ ]:
r2 = r2_score(df['jv_default_pce'], df['predicted_pce'])
mae = mean_absolute_error(df['jv_default_pce'], df['predicted_pce'])

fig = px.scatter(
    df, 
    x='jv_default_pce', 
    y='predicted_pce', 
    color='cell_architecture', 
    opacity=0.6,
    hover_data=['composition_long_form'],
    title=f"Parity Plot: Experimental vs. Predicted PCE (R²={r2:.3f}, MAE={mae:.2f}%)",
    labels={'jv_default_pce': 'Experimental PCE (%)', 'predicted_pce': 'Predicted PCE (%)'}
)
fig.add_shape(type="line", x0=0, y0=0, x1=25, y1=25, line=dict(color="Red", dash="dash"))
fig.update_layout(width=800, height=800, title_x=0.5)
fig.show()

## 6. Residual Analysis: Systematic Biases
Residual analysis helps identify if the model favors certain architectures.

**Observation:** The residuals are roughly centered at zero for both n-i-p and p-i-n, suggesting that the model doesn't have a fundamental bias towards one architecture. However, the n-i-p distribution is wider, likely because it has more diverse HTL/ETL combinations in the dataset.

In [ ]:
fig = px.histogram(
    df, 
    x='residual', 
    color='cell_architecture', 
    marginal="box", 
    barmode="overlay",
    title="Residual Distribution: Uncovering Systematic Errors",
    labels={'residual': 'Error (Experimental - Predicted, %)'}
)
fig.update_layout(width=900, height=600, title_x=0.5)
fig.show()

## 7. Physics-ML Interaction: The Band Gap Manifold
This 2D density plot reveals the 'Effciency-Gap' manifold explored by the literature.

**Observation:** Research is heavily concentrated in the $1.5\text{--}1.6\text{ eV}$ region, which corresponds to the triple-cation (MA/FA/Cs) perovskites. This is the 'efficiency island' that provides the best tradeoff between light absorption and thermodynamic stability.

## 8. Limitations and Scientific Assumptions

### 8.1. Missing Processing Descriptors
*   **The Process Paradox:** Two devices with the same chemical composition and HTL can have vastly different PCEs due to spin-coating speed, annealing temperature, or solvent choice (e.g., DMSO vs. DMF). These are currently 'latent variables' not included in the model, limiting the $R^2$ cap.
*   **Morphology:** ML cannot 'see' the grain size or crystallinity from the composition alone, which are critical for charge carrier mobility.

### 8.2. Data Bias
*   **The 'Hero Cell' Bias:** Negative results (failed devices) are rarely published. This 'Publication Bias' means the model is trained on an artificially successful subset of the chemical space, leading to optimistic predictions in unexplored regions.
*   **Protocol Variance:** Different labs use different J-V scan speeds and directions (Forward vs. Reverse), which can introduce hysteresis-related errors in the reported PCE.

In [ ]:
fig = px.scatter(
    df, 
    x='band_gap', 
    y='jv_default_pce', 
    color='cell_architecture', 
    opacity=0.4,
    hover_data=['composition_long_form'],
    title="The Efficiency-Band Gap Manifold",
    labels={'band_gap': 'Band Gap (eV)', 'jv_default_pce': 'Experimental PCE (%)'}
)
fig.add_density_contour(x=df['band_gap'], y=df['jv_default_pce'], colorscale='Greens', showlabels=True)
fig.add_shape(type="rect", x0=1.45, y0=18, x1=1.65, y1=26, fillcolor="green", opacity=0.1, line_width=0, 
              annotation_text="High Efficiency Region")
fig.update_layout(width=900, height=700, title_x=0.5)
fig.show()